# EII 7435 — Simulación Avanzada
## Generadores de Números Aleatorios

**Instrucciones:** Ejecuta cada celda en orden. Las celdas con `#TU CÓDIGO AQUÍ` requieren que completes el código.

---
## Configuración inicial
Ejecuta esta celda para importar las librerías necesarias.

In [ ]:
# Librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import google.colab
from google.colab import output
output.enable_custom_widget_manager()

---
# PARTE 1: Generación de Números Aleatorios

En este curso construimos modelos para representar la realidad considerando la incertidumbre asociada a esta realidad. Para ello requerimos generar números aleatorios en el computador.

### Contexto

Los fenómenos aleatorios son representados en los modelos de simulación mediante variables aleatorias con distribuciones de probabilidad pre-especificadas. Durante la corrida de simulación se genera un valor aleatorio proveniente de la distribución especificada cada vez que se requiere una realización de la variable aleatoria correspondiente (es decir cada vez que ocurre una instancia del fenómeno).

Prácticamente todos los métodos utilizados para generar valores aleatorios provenientes de una distribución específica, usan como “materia prima” uno o más valores provenientes de una distribución uniforme, $U[0,1]$.

> Los valores aleatorios provenientes de una distribución $U[0,1]$ se denominan **Números Aleatorios**.

La mayoría de los generadores de números aleatorios usados en la actualidad utilizan **métodos aritméticos secuenciales** en que cada número generado se obtiene a partir de uno o varios números generados anteriormente. Más específicamente se tiene que:

$$x_i = f(x_{i-1}, x_{i-2}, \ldots, x_l)$$

La principal objeción a este método es que los números generados no son realmente aleatorios, puesto que están determinados por el valor inicial. Sin embargo, existen buenos generadores aritméticos que generan una secuencia de números que aparentan ser una realización de una secuencia IID de variables $U[0,1]$, en el sentido que no es posible rechazar una prueba de hipótesis al respecto, a ningún nivel de confianza razonable. Por esta razón los números provenientes de un generador aritmético son también denominados **pseudoaleatorios**.


### Método de Congruencia Lineal (LCG)

La mayoría de los generadores aritméticos usados en la actualidad usan el
**Método de Congruencia Lineal de Lehmer**. Los generadores de congruencia lineal (LCG), utilizan la formula recursiva:

$$z_{i} = (a \cdot z_{i-1} + c)(\bmod m)$$

$$u_i = \frac{z_i}{m}$$

Donde:
- $z_0, z_1, z_2, \dots$ es una secuencia de números enteros
- $m$ se denomina **módulo** (típicamente un valor extremadamente grande, e.g. $2^{31}$)
- $a$ se denomina **multiplicador** $(a < m)$
- $c$ se denomina **incremento** $(c < m)$
- $a$, $c$ y $m$ son enteros no negativos
- el valor inicial $z_0$ se denomina **semilla** $(z_0 < m)$

> La función $\bmod$ corresponde al remanente obtenido luego de dividir por $m$. $$ Ej. \quad 11 \bmod 8 = 3$$

### Ejemplo: Cálculo manual de un LCG

Considera $z_0 = 27$, $a = 17$, $c = 43$, $m = 100$.

Calcula $z_1, \ldots, z_7$ a mano, puedes verificar tus resultados usando el código en la siguiente celda.

>`print()` es una función integrada de Python que muestra texto o valores en la pantalla. Puedes pasarle uno o más argumentos separados por comas. Por ejemplo, `print('Hola', 2+3)` imprime `Hola 5`. Usaremos `print()` a lo largo de este documento para visualizar resultados intermedios.

In [ ]:
# La siguiente función implementa un Generador de Congruencia Lineal (LCG).
# Recibe los parámetros a, c, m y la semilla z0.
# Retorna la secuencia de valores zi y ui hasta i = n.
def lcg(z0, a, c, m, n):
    z = np.zeros(n + 1, dtype=np.int64)
    z[0] = z0
    for i in range(n):
        z[i + 1] = (a * z[i] + c) % m
    u = z / m
    return z, u

z, u = lcg(z0=27, a=17, c=43, m=100, n=7) # Parámetros del ejemplo
print(f'z: {z}\n')
print(f'u: {u}\n')

for i in range(1, 8):
    print(f'  z_{i} = (17 × {z[i-1]} + 43) mod 100 = {z[i]}  →  u_{i} = {z[i]}/100 = {u[i]:.2f}')

### Preguntas

1. ¿Cuál es el **máximo** de números aleatorios distintos que puedes obtener con $m=100$?
2. ¿Qué otras características tendría un **buen** generador con $m=100$?

### Ejercicio: Prueba con otros parámetros

Usa $z_0 = 19$, $a = 22$, $c = 4$, $m = 63$. ¿Qué observas?

In [ ]:
# TU CÓDIGO AQUÍ
# Usa la función lcg() definida arriba con los nuevos parámetros. Imprime algunos valores y ve si observas algún comportamiento interesante.

### Exploración: Longitud del ciclo

La **longitud del ciclo** (o período) de un LCG es crucial. Un buen generador debe tener un período largo antes de repetirse.

In [ ]:
# La siguiente función calcula la **longitud del ciclo** de un LCG.
# Genera valores sucesivos y registra en qué iteración apareció cada uno.
# Cuando un valor se repite, la diferencia entre las dos apariciones es la longitud del ciclo.
def encontrar_ciclo(z0, a, c, m, max_iter=10000):
    z = z0
    vistos = {}
    for i in range(max_iter):
        z = (a * z + c) % m
        if z in vistos:
            return i - vistos[z]
        vistos[z] = i
    return max_iter  # No encontró ciclo

# Comparar diferentes parámetros
configs = [
    {'z0': 27, 'a': 17, 'c': 43, 'm': 100},
    {'z0': 19, 'a': 22, 'c': 4,  'm': 63},
    {'z0': 1,  'a': 7,  'c': 0,  'm': 32},
    {'z0': 1,  'a': 5,  'c': 3,  'm': 16},
]

print(f'{"z0":>4} {"a":>4} {"c":>4} {"m":>6}  │  Ciclo')
print('─' * 40)
for cfg in configs:
    ciclo = encontrar_ciclo(**cfg)
    print(f'{cfg["z0"]:>4} {cfg["a"]:>4} {cfg["c"]:>4} {cfg["m"]:>6}  │  {ciclo}')

---
# PARTE 2: Calidad de un Generador — El generador RANDU

**RANDU** es un Generador de Congruencia Lineal introducido a principios de los 60 por IBM para ser utilizado en sus computadoras. RANDU es considerado **uno de los peores Generadores de números aleatorios alguna vez diseñado**.
Para entender la causa de su lamentable fama, analizaremos sus características.

Los parámetros del generador RANDU son los siguientes:

$$a = 65539, \quad c = 0, \quad m = 2^{31}$$

Además se sabe que el número 1 es una semilla que genera un ciclo suficientemente largo de números aleatorios.

Analizaremos por qué es tan malo mediante el **análisis espectral**.

## A) Generación de secuencias
Primero generaremos 2000 números con RANDU y 2000 con un buen generador (Mersenne Twister de NumPy).

In [ ]:
# Parámetros RANDU
a_randu = 65539
c_randu = 0
m_randu = 2**31
semilla_randu = 1

N = 2000

# Generar secuencia RANDU
z_randu, u_randu = lcg(z0=semilla_randu, a=a_randu, c=c_randu, m=m_randu, n=N)
u_randu = u_randu[1:]  # Descartar u_0 (la semilla dividida por m)

# Generar secuencia con Mersenne Twister (generador de buena calidad)
rng = np.random.default_rng(seed=7435)
u_bueno = rng.random(N)

print(f'RANDU  — primeros 10 valores: {np.round(u_randu[:10], 6)}')
print(f'NumPy  — primeros 10 valores: {np.round(u_bueno[:10], 6)}')

## B) Comparación en 2D — Análisis Espectral

Graficamos pares ordenados $(u_i, u_{i+1})$. Un buen generador debe llenar el cuadrado unitario de forma uniforme.

> Se pueden descartar el primer y último par.

**¿Qué podemos concluir sobre la calidad de estos generadores mediante el análisis en 2D?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mersenne Twister
axes[0].scatter(u_bueno[:-1], u_bueno[1:], s=1, alpha=0.5, c='steelblue')
axes[0].set_title('Mersenne Twister (NumPy)', fontsize=14)
axes[0].set_xlabel('$u_i$')
axes[0].set_ylabel('$u_{i+1}$')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_aspect('equal')

# RANDU
axes[1].scatter(u_randu[:-1], u_randu[1:], s=1, alpha=0.5, c='crimson')
axes[1].set_title('RANDU', fontsize=14)
axes[1].set_xlabel('$u_i$')
axes[1].set_ylabel('$u_{i+1}$')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_aspect('equal')

plt.suptitle('Análisis Espectral 2D: Pares $(u_i, u_{i+1})$', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## C) Comparación en 3D

Ahora graficamos  $(u_i, u_{i+1}, u_{i+2})$.

**Usa el mouse para rotar los gráficos 3D y vuelve a evaluar la calidad de los generadores.**

In [ ]:
# --- Gráfico 3D interactivo: Mersenne Twister ---
fig_bueno = go.Figure(data=[go.Scatter3d(
    x=u_bueno[:-2],
    y=u_bueno[1:-1],
    z=u_bueno[2:],
    mode='markers',
    marker=dict(size=1.5, color='steelblue', opacity=0.6)
)])

fig_bueno.update_layout(
    title='Mersenne Twister —  (u(i), u(i+1), u(i+2))',
    scene=dict(
        xaxis_title='u(i)',
        yaxis_title='u(i+1)',
        zaxis_title='u(i+2)',
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        zaxis=dict(range=[0, 1]),
    ),
    width=700, height=600,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig_bueno.show()

In [ ]:
# --- Gráfico 3D interactivo: RANDU ---
fig_randu = go.Figure(data=[go.Scatter3d(
    x=u_randu[:-2],
    y=u_randu[1:-1],
    z=u_randu[2:],
    mode='markers',
    marker=dict(size=1.5, color='crimson', opacity=0.6)
)])

fig_randu.update_layout(
    title='RANDU — (u_i, u(i+1), u(i+2))',
    scene=dict(
        xaxis_title='u(i)',
        yaxis_title='u(i+1)',
        zaxis_title='u(i+2)',
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        zaxis=dict(range=[0, 1]),
    ),
    width=700, height=600,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig_randu.show()

---
## D) Investiga: Mersenne Twister

En este documento utilizamos el generador incluido en NumPy llamado Mersenne Twister. Este generador es también utilizado por defecto en software como Excel, Simio y R.

Investiga y contesta las siguientes preguntas.

- ¿Cuál es el ciclo de Mersenne Twister?
- La garantía de independencia (i.i.d.) aplica solo dentro de una misma secuencia de números aleatorios. Si necesitas realizar múltiples réplicas independientes, ¿cómo podrías lograrlo aprovechando un ciclo como el de Mersenne Twister?

---
## Resumen

1. Los generadores de números pseudoaleatorios son **determinísticos**, es decir, la secuencia completa está determinada por la semilla.

2. Un generador puede *parecer* aleatorio en una dimensión pero tener **estructura oculta** en dimensiones superiores.

3. El **análisis espectral** es una herramienta poderosa para detectar problemas.

4. Los generadores modernos (como Mersenne Twister) son mucho mejores que RANDU, pero la lección sigue siendo relevante: **siempre debemos validar nuestras herramientas**.
